In [1]:
import math
import pandas as pd

from netids.adapters.suricata import suricata_eve_to_features
from netids.features.schema import validate_feature_matrix

# Hai sự kiện thực tế lấy từ VM1, đã bỏ IP và payload.
events = [
    {
        "event_type": "flow",
        "proto": "TCP",
        "flow": {
            "start": "2026-10-25T10:35:24.229017+0000",
            "end": "2026-10-25T10:35:24.278452+0000",
            "age": 0,
            "pkts_toserver": 2,
            "pkts_toclient": 1,
            "bytes_toserver": 182,
            "bytes_toclient": 154,
            "state": "new",
            "reason": "timeout",
        },
    },
    {
        "event_type": "flow",
        "proto": "UDP",
        "flow": {
            "start": "2026-10-25T11:02:21.724721+0000",
            "end": "2026-10-25T11:02:22.155155+0000",
            "age": 1,
            "pkts_toserver": 4,
            "pkts_toclient": 0,
            "bytes_toserver": 344,
            "bytes_toclient": 0,
            "state": "new",
            "reason": "timeout",
        },
    },
]

features = [suricata_eve_to_features(event) for event in events]

# Kiểm chứng độc lập các giá trị phải tính được.
for feature, (is_tcp, duration, packets, bytes_) in zip(
    features,
    [
        (1, 0.049435, 3, 336),
        (0, 0.430434, 4, 344),
    ],
):
    expected = {
        "protocol_tcp": is_tcp,
        "duration_log": math.log1p(duration),
        "packets_log": math.log1p(packets),
        "bytes_log": math.log1p(bytes_),
    }

    for name, expected_value in expected.items():
        assert math.isclose(
            feature[name],
            expected_value,
            rel_tol=1e-12,
            abs_tol=1e-12,
        ), f"Mismatch: {name}"

X = validate_feature_matrix(pd.DataFrame(features))

assert X.shape == (2, 4)

print("=== REAL SURICATA EVE FEATURE VALIDATION ===")
print(X.to_string(index=False))
print("\nPASS: TWO REAL EVE SAMPLES VALIDATED")
print("No ML inference performed.")

=== REAL SURICATA EVE FEATURE VALIDATION ===
 protocol_tcp  duration_log  packets_log  bytes_log
            1      0.048252     1.386294   5.820083
            0      0.357978     1.609438   5.843544

PASS: TWO REAL EVE SAMPLES VALIDATED
No ML inference performed.


In [2]:
from pathlib import Path
import hashlib
import pandas as pd

from netids.adapters.suricata import suricata_eve_to_features
from netids.features.schema import validate_feature_matrix
from netids.models.artifacts import load_ml_artifact

print("=== SURICATA EVE -> RF V0.2 DRY-RUN ===")

artifact_dir = Path(
    r"D:\suricata-ai-project\artifacts\models\ctu13_rf_purged_v0_2"
)

model_path = artifact_dir / "pipeline.joblib"

expected_sha256 = (
    "dc727edd3589666d4c8e2bd4bdbbe2a85802003db44aa5dd37cfaf8ecf44b62a"
)

# 1. Kiểm tra artifact đã đóng băng
hasher = hashlib.sha256()

with model_path.open("rb") as file:
    for block in iter(lambda: file.read(1024 * 1024), b""):
        hasher.update(block)

assert hasher.hexdigest() == expected_sha256, (
    "Model SHA256 mismatch: stop inference."
)

# 2. Tạo đặc trưng từ hai event thật ở cell trước
assert len(events) == 2

X_dryrun = validate_feature_matrix(
    pd.DataFrame([
        suricata_eve_to_features(event)
        for event in events
    ])
)

# 3. Tải RF v0.2 nguyên bản — không huấn luyện lại
model, metadata = load_ml_artifact(artifact_dir)

classes = list(model.classes_)
assert set(classes) == {0, 1}

positive_index = classes.index(1)
scores = model.predict_proba(X_dryrun)[:, positive_index]

# 4. Chỉ hiển thị kết quả thử nghiệm
result = pd.DataFrame({
    "protocol": [event["proto"] for event in events],
    "botnet_score": scores,
    "predicted_class_at_0_5": (scores >= 0.5).astype(int),
})

print(result.to_string(index=False))
print("\nPASS: EVE -> RF V0.2 DRY-RUN COMPLETED")
print("No model retraining.")
print("No alert or Suricata rule generated.")

=== SURICATA EVE -> RF V0.2 DRY-RUN ===
protocol  botnet_score  predicted_class_at_0_5
     TCP      0.277966                       0
     UDP      0.672917                       1

PASS: EVE -> RF V0.2 DRY-RUN COMPLETED
No model retraining.
No alert or Suricata rule generated.


In [5]:
import json
import hashlib
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from netids.adapters.suricata import suricata_eve_to_features
from netids.features.schema import validate_feature_matrix
from netids.models.artifacts import load_ml_artifact

print("=== EVE BATCH -> RF V0.2 DRY-RUN ===")

root = Path(r"D:\suricata-ai-project")

data_path = (
    root / "data" / "processed"
    / "eve-flow-batch-sanitized.jsonl"
)

model_dir = (
    root / "artifacts" / "models"
    / "ctu13_rf_purged_v0_2"
)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

assert file_sha256(data_path) == (
    "b7afdcf335e168f4828d89ab52de9c2ec53ed6b83c87b91bb360bd31a95d9bbb"
)

assert file_sha256(model_dir / "pipeline.joblib") == (
    "dc727edd3589666d4c8e2bd4bdbbe2a85802003db44aa5dd37cfaf8ecf44b62a"
)

features = []
protocols = []
rejected = Counter()
total = 0

with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        total += 1

        try:
            event = json.loads(line)
            feature = suricata_eve_to_features(event)

        except (ValueError, TypeError, KeyError, OverflowError) as exc:
            rejected[type(exc).__name__] += 1
            continue

        features.append(feature)
        protocols.append(event["proto"].upper())

assert total == 72
assert features, "No valid flow available for inference"

X = validate_feature_matrix(pd.DataFrame(features))

model, metadata = load_ml_artifact(model_dir)

assert set(model.classes_) == {0, 1}

positive_index = list(model.classes_).index(1)

scores = model.predict_proba(X)[:, positive_index]

assert np.isfinite(scores).all()

results = pd.DataFrame({
    "protocol": protocols,
    "botnet_score": scores,
    "predicted_botnet": (scores >= 0.5).astype(int),
})

summary = (
    results.groupby("protocol")
    .agg(
        valid_flows=("botnet_score", "size"),
        predicted_botnet=("predicted_botnet", "sum"),
        median_score=("botnet_score", "median"),
        min_score=("botnet_score", "min"),
        max_score=("botnet_score", "max"),
    )
)

print("Total JSONL records:", total)
print("Valid flows:", len(results))
print("Rejected flows:", sum(rejected.values()))
print("Rejection reasons:", dict(rejected))

print("\n=== RESULTS BY PROTOCOL ===")
print(summary.round(6).to_string())

print("\nPASS: EVE BATCH DRY-RUN COMPLETED")
print("No model retraining.")
print("No alert or Suricata rule generated.")

=== EVE BATCH -> RF V0.2 DRY-RUN ===
Total JSONL records: 72
Valid flows: 72
Rejected flows: 0
Rejection reasons: {}

=== RESULTS BY PROTOCOL ===
          valid_flows  predicted_botnet  median_score  min_score  max_score
protocol                                                                   
TCP                30                 8      0.277966   0.000000   0.825390
UDP                42                20      0.436443   0.024168   0.832283

PASS: EVE BATCH DRY-RUN COMPLETED
No model retraining.
No alert or Suricata rule generated.
